In [56]:
from langgraph.graph import StateGraph,START,END
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import TypedDict

In [ ]:
model = ChatOllama(
    model = 'deepseek-r1:1.5b',
    temperature=1.4
)

parser = StrOutputParser()

In [58]:
class LLMState(TypedDict):
    question: str
    answer: str

In [59]:
def llm_qa(state: LLMState) -> LLMState:
    question = PromptTemplate(
        template= 
        '''you are an AI MODEL Which will answer to general questions asked by user.
        question: {question}''',
        input_variables=['question']
    )
    final_prompt= question.invoke(state['question'])
    chain = model | parser
    result = chain.invoke(final_prompt)
    state['answer'] = result
    return state

In [60]:
graph = StateGraph(LLMState)

graph.add_node("llm_qa",llm_qa)

graph.add_edge(START,'llm_qa')
graph.add_edge("llm_qa",END)

In [61]:
compilation = graph.compile()

In [66]:
initial_state = {'question' : "how far is the moon from the earth with exact distance?."}

result=compilation.invoke(initial_state)

In [67]:
print(result)

{'question': 'how far is the moon from the earth with exact distance?.', 'answer': '\n\nThe average distance between the Earth and the Moon is approximately 384,000 kilometers. This value accounts for the variation in distance over different months but represents the typical figure used in such calculations.\n\n**Answer:** The Earth-Moon system averages about 384,000 kilometers apart during their orbit around the Sun.'}
